# 6-3. 기준지표 교체 — 7분위 CRPS + 커버리지 + MSE

5-1은 리샘플링 CRPS+bootstrap+최대오차로 변수별 모델을 골랐다. 여기선 지표를 3개(7분위 CRPS/커버리지/MSE)로 바꿔 같은 16개+E3 후보를 재채점하고 5-1과 비교한다.

- **7분위 CRPS**: 분포 적합도 (4-4 분위수근사법)
- **커버리지**: 구간이 90%를 지키는지
- **MSE**: 점예측 정확도

세 순위를 Borda로 합쳐 최종 선정한다.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

COLS = ["X1_l1", "X2", "X3_l1", "X4", "X5_l1", "X6_l1"]
LABELS = {
    "X1_l1": "X1", "X2": "X2", "X3_l1": "X3",
    "X4": "X4", "X5_l1": "X5", "X6_l1": "X6",
}
H = 4
INITIAL_CALIB = 15
MIN_TRAIN = 20
LAG = 2
E3 = ["arima_aic", "gp_matern", "gp_rbf"]
QUANTILE_LEVELS = [0.1, 0.2, 0.3, 0.5, 0.7, 0.8, 0.9]

FAMILY_LABELS = {
    "reg_ols": "회귀-OLS", "reg_ridge": "회귀-Ridge", "reg_lasso": "회귀-Lasso", "reg_elastic": "회귀-ElasticNet",
    "arima_aic": "시계열-ARIMA(AIC)", "arima_bic": "시계열-ARIMA(BIC)", "theta": "시계열-Theta",
    "gp_rbf": "GP-RBF", "gp_matern": "GP-Matern", "gp_rq": "GP-RQ", "gp_rbf_linear": "GP-RBF+Linear",
    "naive": "baseline (naive)", "gbm_lag": "트리기반 (GBM)",
    "rf_lag": "트리기반 (RandomForest)", "knn_lag": "k-NN", "holt_damped": "시계열-Holt(감쇠추세)",
}
CANDIDATES = list(FAMILY_LABELS.keys())

raw = pd.read_csv("../../data/gold/gold_major_variables.csv")
raw = raw[raw["quarter"].notna()].reset_index(drop=True)

err_42 = pd.read_pickle("../cache/cache_nb4_2_err_df.pkl")
err_32 = pd.read_pickle("../cache/cache_nb3_2_err_df.pkl")
extra = err_32[err_32["model"].isin(["naive", "gbm_lag"])]
err_df = pd.concat([err_42, extra], ignore_index=True)


def rng_for(name):
    return np.random.default_rng(int.from_bytes(name.encode(), "little") % (2**32))


print(f"베이스 캐시: {err_df['model'].nunique()}개 모델 x {err_df['var'].nunique()}개 변수")


베이스 캐시: 13개 모델 x 6개 변수


## 0. 신규 후보 3개(RF/Holt/kNN) 백테스트 -- 6개 변수 전체

5-1의 16개 후보를 재사용하기 위해, 캐시에 없는 RF/Holt/kNN만 6개 변수로 재계산.

In [2]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from statsmodels.tsa.holtwinters import ExponentialSmoothing


def _lag_xy(values, lag=LAG):
    X = np.array([values[i:i + lag] for i in range(len(values) - lag)])
    y = values[lag:]
    return X, y


def _recursive_forecast(model_fit_fn, values, h=H, lag=LAG):
    X, y = _lag_xy(values, lag)
    model = model_fit_fn(X, y)
    hist = list(values[-lag:])
    out = []
    for _ in range(h):
        x = np.array(hist[-lag:]).reshape(1, -1)
        yp = model.predict(x)[0]
        out.append(yp)
        hist.append(yp)
    return np.array(out)


def fc_rf(values, h=H):
    return _recursive_forecast(lambda X, y: RandomForestRegressor(n_estimators=200, max_depth=4, random_state=0).fit(X, y), values, h)


def fc_knn(values, h=H):
    return _recursive_forecast(lambda X, y: KNeighborsRegressor(n_neighbors=3).fit(X, y), values, h)


def fc_holt(values, h=H):
    fit = ExponentialSmoothing(values, trend="add", damped_trend=True, initialization_method="estimated").fit()
    return np.asarray(fit.forecast(h))


NEW_FUNCS = {"rf_lag": fc_rf, "knn_lag": fc_knn, "holt_damped": fc_holt}

new_rows = []
for v in COLS:
    vals = raw[["quarter", v]].dropna()[v].values.astype(float)
    for origin in range(MIN_TRAIN, len(vals) - H + 1):
        train, actual = vals[:origin], vals[origin:origin + H]
        for name, fn in NEW_FUNCS.items():
            try:
                fc = fn(train, H)
            except Exception:
                fc = np.repeat(train[-1], H)
            err = actual - fc
            for h in range(1, H + 1):
                new_rows.append({"var": v, "model": name, "origin": origin, "h": h,
                                  "err_t": err[h - 1], "fc_t": fc[h - 1], "actual_orig": actual[h - 1]})
err_df = pd.concat([err_df, pd.DataFrame(new_rows)], ignore_index=True)
print(f"신규 3개 x 6개 변수 백테스트 완료({len(new_rows)}행 추가) -- 최종 후보 {len(CANDIDATES)}개 x 변수 {len(COLS)}개")


신규 3개 x 6개 변수 백테스트 완료(2676행 추가) -- 최종 후보 16개 x 변수 6개


## 1. 3개 지표 계산 함수

(모델, 변수)별 22개 채점 origin 평균값을 반환한다.

In [3]:
def pinball_loss(q_pred, y, tau):
    return (y - q_pred) * tau if y >= q_pred else (q_pred - y) * (1 - tau)


def crps_quantile_approx_for(name, var, n_samples=500):
    rng = rng_for(name)
    e = err_df[(err_df["model"] == name) & (err_df["var"] == var)]
    origins = sorted(e["origin"].unique())
    var_std = e["actual_orig"].std()
    scores = []
    for i in range(INITIAL_CALIB, len(origins)):
        calib = e[e["origin"].isin(origins[:i])]
        test = e[e["origin"] == origins[i]]
        for h in range(1, H + 1):
            calib_resid_h = calib[calib["h"] == h]["err_t"].values
            row = test[test["h"] == h].iloc[0]
            samples = row["fc_t"] + rng.choice(calib_resid_h, size=n_samples, replace=True)
            q_preds = np.quantile(samples, QUANTILE_LEVELS)
            losses = [pinball_loss(q, row["actual_orig"], tau) for q, tau in zip(q_preds, QUANTILE_LEVELS)]
            scores.append(2 * np.mean(losses) / var_std)
    return np.mean(scores)


def coverage_for(name, var, alpha=0.10):
    rng = rng_for(name)
    e = err_df[(err_df["model"] == name) & (err_df["var"] == var)]
    origins = sorted(e["origin"].unique())
    hits = []
    for i in range(INITIAL_CALIB, len(origins)):
        calib = e[e["origin"].isin(origins[:i])]
        test = e[e["origin"] == origins[i]]
        for h in range(1, H + 1):
            calib_resid_h = calib[calib["h"] == h]["err_t"].values
            row = test[test["h"] == h].iloc[0]
            samples = row["fc_t"] + rng.choice(calib_resid_h, size=500, replace=True)
            lo, hi = np.quantile(samples, [alpha / 2, 1 - alpha / 2])
            hits.append(lo <= row["actual_orig"] <= hi)
    return np.mean(hits) * 100


def mse_norm_for(name, var):
    e = err_df[(err_df["model"] == name) & (err_df["var"] == var)]
    origins = sorted(e["origin"].unique())
    var_std = e["actual_orig"].std()
    sq_errs = []
    for i in range(INITIAL_CALIB, len(origins)):
        test = e[e["origin"] == origins[i]]
        for h in range(1, H + 1):
            row = test[test["h"] == h].iloc[0]
            sq_errs.append((row["err_t"] / var_std) ** 2)
    return np.mean(sq_errs)


## 2. E3 앙상블 버전 (풀링 방식은 5-1과 동일)


In [4]:
def ensemble_crps_quantile_approx(models, var, n_each=200):
    rngs = {m: rng_for(m) for m in models}
    sub = {m: err_df[(err_df["model"] == m) & (err_df["var"] == var)] for m in models}
    var_std = sub[models[0]]["actual_orig"].std()
    origins = sorted(sub[models[0]]["origin"].unique())
    scores = []
    for i in range(INITIAL_CALIB, len(origins)):
        for h in range(1, H + 1):
            pooled, actual_orig = [], None
            for m in models:
                g = sub[m]
                calib = g[g["origin"].isin(origins[:i])]
                test = g[g["origin"] == origins[i]]
                calib_resid_h = calib[calib["h"] == h]["err_t"].values
                row = test[test["h"] == h].iloc[0]
                actual_orig = row["actual_orig"]
                samples = row["fc_t"] + rngs[m].choice(calib_resid_h, size=n_each, replace=True)
                pooled.append(samples)
            pooled = np.concatenate(pooled)
            q_preds = np.quantile(pooled, QUANTILE_LEVELS)
            losses = [pinball_loss(q, actual_orig, tau) for q, tau in zip(q_preds, QUANTILE_LEVELS)]
            scores.append(2 * np.mean(losses) / var_std)
    return np.mean(scores)


def ensemble_coverage(models, var, alpha=0.10, n_each=200):
    rngs = {m: rng_for(m) for m in models}
    sub = {m: err_df[(err_df["model"] == m) & (err_df["var"] == var)] for m in models}
    origins = sorted(sub[models[0]]["origin"].unique())
    hits = []
    for i in range(INITIAL_CALIB, len(origins)):
        for h in range(1, H + 1):
            pooled, actual_orig = [], None
            for m in models:
                g = sub[m]
                calib = g[g["origin"].isin(origins[:i])]
                test = g[g["origin"] == origins[i]]
                calib_resid_h = calib[calib["h"] == h]["err_t"].values
                row = test[test["h"] == h].iloc[0]
                actual_orig = row["actual_orig"]
                pooled.append(row["fc_t"] + rngs[m].choice(calib_resid_h, size=n_each, replace=True))
            pooled = np.concatenate(pooled)
            lo, hi = np.quantile(pooled, [alpha / 2, 1 - alpha / 2])
            hits.append(lo <= actual_orig <= hi)
    return np.mean(hits) * 100


def ensemble_mse_norm(models, var):
    sub = {m: err_df[(err_df["model"] == m) & (err_df["var"] == var)] for m in models}
    var_std = sub[models[0]]["actual_orig"].std()
    origins = sorted(sub[models[0]]["origin"].unique())
    sq_errs = []
    for i in range(INITIAL_CALIB, len(origins)):
        for h in range(1, H + 1):
            fc_mean, actual_orig = [], None
            for m in models:
                g = sub[m]
                test = g[g["origin"] == origins[i]]
                row = test[test["h"] == h].iloc[0]
                actual_orig = row["actual_orig"]
                fc_mean.append(row["fc_t"])
            sq_errs.append(((actual_orig - np.mean(fc_mean)) / var_std) ** 2)
    return np.mean(sq_errs)


## 3. 변수별 x 후보별 3개 지표 전부 계산


In [5]:
rows = []
for v in COLS:
    for name in CANDIDATES:
        rows.append({
            "var": v, "_key": name, "모델": FAMILY_LABELS[name],
            "CRPS(7분위)": crps_quantile_approx_for(name, v),
            "커버리지(%)": coverage_for(name, v),
            "MSE(정규화)": mse_norm_for(name, v),
        })
    rows.append({
        "var": v, "_key": "E3", "모델": "E3(공용앙상블)",
        "CRPS(7분위)": ensemble_crps_quantile_approx(E3, v),
        "커버리지(%)": ensemble_coverage(E3, v),
        "MSE(정규화)": ensemble_mse_norm(E3, v),
    })
metric_df = pd.DataFrame(rows)
print(f"{len(metric_df)}행 계산 완료 (변수 {metric_df['var'].nunique()}개 x 후보 {metric_df['_key'].nunique()}개)")


102행 계산 완료 (변수 6개 x 후보 17개)


## 4. 결합순위(Borda) -- 3개 지표를 하나로

CRPS(7분위)/`|커버리지-90|`/MSE 각각 순위를 매겨 평균 -- 낮을수록 좋은 후보.

In [6]:
def borda_rank(var_df):
    d = var_df.copy()
    d["rank_crps"] = d["CRPS(7분위)"].rank()
    d["rank_cov"] = (d["커버리지(%)"] - 90).abs().rank()
    d["rank_mse"] = d["MSE(정규화)"].rank()
    d["결합순위"] = (d["rank_crps"] + d["rank_cov"] + d["rank_mse"]) / 3
    return d.sort_values("결합순위").reset_index(drop=True)


ranked = {v: borda_rank(metric_df[metric_df["var"] == v]) for v in COLS}
for v in COLS:
    print(f"--- {LABELS[v]} ---")
    display(ranked[v][["모델", "CRPS(7분위)", "커버리지(%)", "MSE(정규화)", "결합순위"]].head(5))


--- X1 ---


,모델,CRPS(7분위),커버리지(%),MSE(정규화),결합순위
0,시계열-ARIMA(AIC),0.674298,82.142857,1.213088,1.833333
1,GP-RQ,0.707748,82.142857,1.296315,3.833333
2,트리기반 (GBM),0.695086,83.333333,1.353805,4.333333
3,E3(공용앙상블),0.702750,79.761905,1.272870,5.666667
4,GP-RBF+Linear,0.713461,80.952381,1.303932,6.000000


--- X2 ---


,모델,CRPS(7분위),커버리지(%),MSE(정규화),결합순위
0,E3(공용앙상블),0.511022,77.272727,0.819477,3.166667
1,시계열-ARIMA(BIC),0.514045,77.272727,0.809576,3.500000
2,GP-Matern,0.513811,77.272727,0.846249,3.833333
3,GP-RBF,0.516441,78.409091,0.863845,4.333333
4,GP-RQ,0.516441,78.409091,0.863845,5.000000


--- X3 ---


...

,모델,CRPS(7분위),커버리지(%),MSE(정규화),결합순위
0,회귀-OLS,0.521831,80.681818,0.774435,2.333333
1,E3(공용앙상블),0.511534,77.272727,0.795407,2.333333
2,GP-Matern,0.514638,75.000000,0.774887,3.666667
3,시계열-ARIMA(AIC),0.517096,75.000000,0.852413,5.333333
4,GP-RBF,0.534364,75.000000,0.845545,6.000000


--- X5 ---


,모델,CRPS(7분위),커버리지(%),MSE(정규화),결합순위
0,GP-RBF,0.652285,83.333333,1.363326,4.500000
1,회귀-ElasticNet,0.708204,85.714286,1.575730,5.500000
2,GP-Matern,0.681902,84.523810,1.483482,5.500000
3,회귀-Lasso,0.683665,84.523810,1.570783,6.166667
4,E3(공용앙상블),0.674320,83.333333,1.479292,6.500000


--- X6 ---


,모델,CRPS(7분위),커버리지(%),MSE(정규화),결합순위
0,회귀-OLS,0.232791,88.0,0.208692,3.166667
1,시계열-Theta,0.220603,84.0,0.189831,4.333333
2,시계열-ARIMA(AIC),0.241925,87.0,0.232845,5.000000
3,트리기반 (RandomForest),0.242234,88.0,0.239638,5.166667
4,시계열-ARIMA(BIC),0.244391,87.0,0.231817,5.333333


## 4-1. Bootstrap 신뢰도 검증

결합순위가 노이즈인지 확인하기 위해 block bootstrap(origin 4개씩, 500회)으로 후보별 1위 비율을 본다.

In [7]:
def crps_q_components_for(name, var, n_samples=500):
    rng = rng_for(name)
    e = err_df[(err_df["model"] == name) & (err_df["var"] == var)]
    origins = sorted(e["origin"].unique())
    var_std = e["actual_orig"].std()
    rows = []
    for i in range(INITIAL_CALIB, len(origins)):
        calib = e[e["origin"].isin(origins[:i])]
        test = e[e["origin"] == origins[i]]
        losses_h = []
        for h in range(1, H + 1):
            calib_resid_h = calib[calib["h"] == h]["err_t"].values
            row = test[test["h"] == h].iloc[0]
            samples = row["fc_t"] + rng.choice(calib_resid_h, size=n_samples, replace=True)
            q_preds = np.quantile(samples, QUANTILE_LEVELS)
            losses = [pinball_loss(q, row["actual_orig"], tau) for q, tau in zip(q_preds, QUANTILE_LEVELS)]
            losses_h.append(2 * np.mean(losses) / var_std)
        rows.append({"origin": origins[i], "crps": np.mean(losses_h)})
    return pd.DataFrame(rows)


def coverage_components_for(name, var, alpha=0.10):
    rng = rng_for(name)
    e = err_df[(err_df["model"] == name) & (err_df["var"] == var)]
    origins = sorted(e["origin"].unique())
    rows = []
    for i in range(INITIAL_CALIB, len(origins)):
        calib = e[e["origin"].isin(origins[:i])]
        test = e[e["origin"] == origins[i]]
        hits_h = []
        for h in range(1, H + 1):
            calib_resid_h = calib[calib["h"] == h]["err_t"].values
            row = test[test["h"] == h].iloc[0]
            samples = row["fc_t"] + rng.choice(calib_resid_h, size=500, replace=True)
            lo, hi = np.quantile(samples, [alpha / 2, 1 - alpha / 2])
            hits_h.append(1.0 if lo <= row["actual_orig"] <= hi else 0.0)
        rows.append({"origin": origins[i], "cov": np.mean(hits_h)})
    return pd.DataFrame(rows)


def mse_components_for(name, var):
    e = err_df[(err_df["model"] == name) & (err_df["var"] == var)]
    origins = sorted(e["origin"].unique())
    var_std = e["actual_orig"].std()
    rows = []
    for i in range(INITIAL_CALIB, len(origins)):
        test = e[e["origin"] == origins[i]]
        sq_h = [(test[test["h"] == h].iloc[0]["err_t"] / var_std) ** 2 for h in range(1, H + 1)]
        rows.append({"origin": origins[i], "mse": np.mean(sq_h)})
    return pd.DataFrame(rows)


def ensemble_crps_q_components(models, var, n_each=200):
    rngs = {m: rng_for(m) for m in models}
    sub = {m: err_df[(err_df["model"] == m) & (err_df["var"] == var)] for m in models}
    var_std = sub[models[0]]["actual_orig"].std()
    origins = sorted(sub[models[0]]["origin"].unique())
    rows = []
    for i in range(INITIAL_CALIB, len(origins)):
        losses_h = []
        for h in range(1, H + 1):
            pooled, actual_orig = [], None
            for m in models:
                g = sub[m]
                calib = g[g["origin"].isin(origins[:i])]
                test = g[g["origin"] == origins[i]]
                calib_resid_h = calib[calib["h"] == h]["err_t"].values
                row = test[test["h"] == h].iloc[0]
                actual_orig = row["actual_orig"]
                pooled.append(row["fc_t"] + rngs[m].choice(calib_resid_h, size=n_each, replace=True))
            pooled = np.concatenate(pooled)
            q_preds = np.quantile(pooled, QUANTILE_LEVELS)
            losses = [pinball_loss(q, actual_orig, tau) for q, tau in zip(q_preds, QUANTILE_LEVELS)]
            losses_h.append(2 * np.mean(losses) / var_std)
        rows.append({"origin": origins[i], "crps": np.mean(losses_h)})
    return pd.DataFrame(rows)


def ensemble_coverage_components(models, var, alpha=0.10, n_each=200):
    rngs = {m: rng_for(m) for m in models}
    sub = {m: err_df[(err_df["model"] == m) & (err_df["var"] == var)] for m in models}
    origins = sorted(sub[models[0]]["origin"].unique())
    rows = []
    for i in range(INITIAL_CALIB, len(origins)):
        hits_h = []
        for h in range(1, H + 1):
            pooled, actual_orig = [], None
            for m in models:
                g = sub[m]
                calib = g[g["origin"].isin(origins[:i])]
                test = g[g["origin"] == origins[i]]
                calib_resid_h = calib[calib["h"] == h]["err_t"].values
                row = test[test["h"] == h].iloc[0]
                actual_orig = row["actual_orig"]
                pooled.append(row["fc_t"] + rngs[m].choice(calib_resid_h, size=n_each, replace=True))
            pooled = np.concatenate(pooled)
            lo, hi = np.quantile(pooled, [alpha / 2, 1 - alpha / 2])
            hits_h.append(1.0 if lo <= actual_orig <= hi else 0.0)
        rows.append({"origin": origins[i], "cov": np.mean(hits_h)})
    return pd.DataFrame(rows)


def ensemble_mse_components(models, var):
    sub = {m: err_df[(err_df["model"] == m) & (err_df["var"] == var)] for m in models}
    var_std = sub[models[0]]["actual_orig"].std()
    origins = sorted(sub[models[0]]["origin"].unique())
    rows = []
    for i in range(INITIAL_CALIB, len(origins)):
        sq_h = []
        for h in range(1, H + 1):
            fc_mean, actual_orig = [], None
            for m in models:
                test = sub[m][sub[m]["origin"] == origins[i]]
                row = test[test["h"] == h].iloc[0]
                actual_orig = row["actual_orig"]
                fc_mean.append(row["fc_t"])
            sq_h.append(((actual_orig - np.mean(fc_mean)) / var_std) ** 2)
        rows.append({"origin": origins[i], "mse": np.mean(sq_h)})
    return pd.DataFrame(rows)


print("컴포넌트(origin별) 함수 정의 완료")


컴포넌트(origin별) 함수 정의 완료


In [8]:
def block_resample(origins, block_len, rng):
    n = len(origins)
    n_blocks = int(np.ceil(n / block_len))
    starts = rng.integers(0, max(1, n - block_len + 1), size=n_blocks)
    out = []
    for s in starts:
        out.extend(origins[s:s + block_len])
    return out[:n]


N_BOOT, BLOCK_LEN = 500, 4


def bootstrap_winrate_for_var(var):
    components = {}
    for name in CANDIDATES:
        c1 = crps_q_components_for(name, var).set_index("origin")
        c2 = coverage_components_for(name, var).set_index("origin")
        c3 = mse_components_for(name, var).set_index("origin")
        components[name] = c1.join(c2).join(c3)
    c1 = ensemble_crps_q_components(E3, var).set_index("origin")
    c2 = ensemble_coverage_components(E3, var).set_index("origin")
    c3 = ensemble_mse_components(E3, var).set_index("origin")
    components["E3"] = c1.join(c2).join(c3)

    names = CANDIDATES + ["E3"]
    origins_v = sorted(components[names[0]].index)
    rng = np.random.default_rng(hash(var) % (2**32))
    win_counts = {n: 0 for n in names}
    for _ in range(N_BOOT):
        resampled = block_resample(origins_v, BLOCK_LEN, rng)
        crps_vals, cov_vals, mse_vals = {}, {}, {}
        for n in names:
            sub = components[n].loc[resampled]
            crps_vals[n] = sub["crps"].mean()
            cov_vals[n] = abs(sub["cov"].mean() * 100 - 90)
            mse_vals[n] = sub["mse"].mean()
        rank_crps = pd.Series(crps_vals).rank()
        rank_cov = pd.Series(cov_vals).rank()
        rank_mse = pd.Series(mse_vals).rank()
        combined = (rank_crps + rank_cov + rank_mse) / 3
        win_counts[combined.idxmin()] += 1
    return {n: round(100 * win_counts[n] / N_BOOT, 1) for n in names}


winrates = {v: bootstrap_winrate_for_var(v) for v in COLS}
for v in COLS:
    top = sorted(winrates[v].items(), key=lambda kv: -kv[1])[:5]
    print(f"--- {LABELS[v]} ---")
    for k, wr in top:
        label = FAMILY_LABELS.get(k, "E3(공용앙상블)")
        print(f"  {label}: {wr}%")


--- X1 ---
  시계열-ARIMA(AIC): 36.0%
  GP-RQ: 25.8%
  트리기반 (GBM): 25.6%
  GP-RBF+Linear: 3.2%
  k-NN: 3.2%
--- X2 ---
  시계열-ARIMA(BIC): 40.2%
  GP-RBF+Linear: 24.4%
  GP-Matern: 16.0%
  GP-RBF: 5.6%
  k-NN: 3.4%
--- X3 ---
  GP-Matern: 61.6%
  E3(공용앙상블): 8.0%
  트리기반 (RandomForest): 6.4%
  baseline (naive): 6.2%
  GP-RBF: 4.4%
--- X4 ---
  E3(공용앙상블): 34.6%
  회귀-OLS: 30.0%
  GP-Matern: 12.8%
  시계열-ARIMA(AIC): 11.0%
  baseline (naive): 5.2%
--- X5 ---
  회귀-Lasso: 33.0%
  GP-RBF: 31.2%
  GP-Matern: 10.4%
  GP-RQ: 10.0%
  회귀-ElasticNet: 5.4%
--- X6 ---
  시계열-Theta: 42.2%
  트리기반 (RandomForest): 24.0%
  회귀-OLS: 19.0%
  트리기반 (GBM): 6.2%
  baseline (naive): 3.0%


## 5. 최종 선정 -- 5-1 vs 6-3(bootstrap 검증 포함)

5-1과 동일 기준: bootstrap 1위 비율 60%+ 또는 E3보다 20%p+ 앞서면 교체.

In [9]:
NB51_FINAL = {
    "X1_l1": "시계열-ARIMA(AIC)", "X2": "시계열-ARIMA(BIC)", "X3_l1": "E3(공용모델 유지)",
    "X4": "E3(공용모델 유지)", "X5_l1": "회귀-Lasso", "X6_l1": "시계열-Theta",
}

MARGIN_WINRATE, GAP_OVER_E3 = 60.0, 20.0

compare_rows = []
for v in COLS:
    wr = winrates[v]
    e3_wr = wr["E3"]
    best_key = max(wr, key=wr.get)
    best_wr = wr[best_key]
    best_label = FAMILY_LABELS.get(best_key, "E3(공용앙상블)")
    winrate_ok = (best_key != "E3") and ((best_wr >= MARGIN_WINRATE) or (best_wr - e3_wr >= GAP_OVER_E3))
    chosen_62 = best_label if winrate_ok else "E3(공용모델 유지)"
    compare_rows.append({
        "변수": LABELS[v],
        "5-1 최종선정": NB51_FINAL[v],
        "6-3 최종선정(bootstrap검증)": chosen_62,
        "6-3 1위 승률(%)": best_wr,
        "6-3 E3 승률(%)": e3_wr,
        "결론 동일한가": "동일" if NB51_FINAL[v] == chosen_62 else "**다름**",
    })
compare_df = pd.DataFrame(compare_rows).set_index("변수")
display(compare_df)


,5-1 최종선정,6-3 최종선정(bootstrap검증),6-3 1위 승률(%),6-3 E3 승률(%),결론 동일한가
변수,,,,,
X1,시계열-ARIMA(AIC),시계열-ARIMA(AIC),36.0,3.0,동일
X2,시계열-ARIMA(BIC),시계열-ARIMA(BIC),40.2,2.8,동일
X3,E3(공용모델 유지),GP-Matern,61.6,8.0,**다름**
X4,E3(공용모델 유지),E3(공용모델 유지),34.6,34.6,동일
X5,회귀-Lasso,회귀-Lasso,33.0,4.2,동일
X6,시계열-Theta,시계열-Theta,42.2,0.0,동일


## 6. 결론 및 남은 한계

Bootstrap 적용 후 5/6 변수가 5-1과 동일하게 수렴 -- 단순 순위차(4/6 다름)는 대부분 노이즈였다.

유일한 차이는 X3_l1(GP-Matern 61.0% vs E3 7.6%) -- 5-1에서 이미 확인한 함정과 같다(승률은 이기지만 90% 튜닝 후 구간폭이 1.11배 더 넓어 결국 E3 유지). 지표를 바꿔도 재현된다는 게 이 현상이 진짜임을 뒷받침한다.

**남은 한계**: 최대오차 체크, 커버리지 공정폭비교 없음.

**종합**: 지표 선택보다 안전장치(bootstrap 등) 유무가 결론 안정성을 더 크게 좌우한다.